[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogoflim/Pesquisa-Operacional-III-A/blob/main/11_Simulacao_em_Tabelas.ipynb)

## **Pesquisa Operacional III-A****Professor:**- Diogo Ferreira de Lima Silva (TEP-UFF)

# Simulação em TabelasNas aulas teóricas, construímos **simulações em tabelas** preenchendo, linha a linha:- um **relógio** (o instante atual da simulação);- uma **fila de eventos** (os eventos futuros já conhecidos);- as **variáveis de estado** do sistema.Este notebook implementa em Python **exatamente a mesma mecânica**, para os dois exemplos vistos em sala:1. a **linha de montagem** com três máquinas sequenciais;2. o **exercício do banco** (uma agência M/M/1).O objetivo é que você veja a tabela do quadro **nascer de um código** — e, ao final, calcule as estatísticas da forma conceitualmente correta. Na próxima aula, passaremos a usar a biblioteca **SimPy**, que automatiza esse mesmo motor de eventos.

---## Parte 1 — A mecânica de um algoritmo de simulaçãoTodo algoritmo de simulação de eventos discretos repete o mesmo ciclo:1. **Encontrar o próximo evento** na fila de eventos (o de menor tempo).2. **Avançar o relógio** até o instante desse evento.3. **Disparar o evento**: atualizar as variáveis de estado e, se for o caso, **agendar novos eventos futuros** na fila.4. **Registrar** o estado na tabela de simulação.5. Voltar ao passo 1, até atingir o **critério de parada**.Vamos importar as bibliotecas que usaremos.

In [ ]:
import pandas as pd
import numpy as np

---## Parte 2 — Exemplo da linha de montagemRelembrando o problema visto em sala:- Um produto passa por **três máquinas sequenciais** $M_1 \\to M_2 \\to M_3$.- $M_1$ **nunca fica ociosa** (há sempre insumo disponível).- Cada máquina processa **uma unidade por vez**.**Variáveis de estado** (duas bastam):- $L_2$: número de unidades em $M_2$ **+** sua fila;- $L_3$: número de unidades em $M_3$ **+** sua fila.**Eventos:**| Evento | Significado ||---|---|| `T1` | $M_1$ termina o processamento || `T2` | $M_2$ termina o processamento || `T3` | $M_3$ termina o processamento |

### 2.1 — Os tempos de processamentoEm sala, usamos uma tabela de números aleatórios já gerada (função `ALEATÓRIOENTRE` do Excel). Vamos usar **os mesmos valores**, para que o resultado do código seja idêntico ao da tabela do quadro.Cada lista abaixo é consumida **na ordem**: a primeira vez que $M_1$ termina, usamos `tempos_M1[0]`; a segunda vez, `tempos_M1[1]`; e assim por diante.

In [ ]:
# Tempos de processamento (mesmos valores da tabela auxiliar usada em sala)
tempos_M1 = [2, 2, 2, 3, 3, 3, 3, 2, 3, 1]
tempos_M2 = [3, 4, 4, 3, 3, 5, 3, 5, 5, 3]
tempos_M3 = [2, 3, 2, 2, 2, 4, 4, 2, 3, 4]

# Conjuntos de onde esses valores vieram (uniforme discreta):
# M1 em {1,2,3}, M2 em {2,3,4}, M3 em {3,4,5}, todos equiprovaveis.

### 2.2 — As regras de cada eventoEsta é a tradução direta da tabela "Efeitos no Estado / Efeitos na fila de eventos" dos slides.Repare em um detalhe importante: a condição para agendar `T2` quando ocorre `T1` é **"$M_2$ estava ociosa"**, ou seja, $L_2 = 0$ **antes** do incremento. Quando $M_2$ já está ocupada, a unidade que chega apenas entra na fila — o término dela será agendado mais tarde, quando a unidade da frente terminar.O mesmo raciocínio vale para `T2 → T3`.

### 2.3 — O motor de simulaçãoA função abaixo é o coração da simulação. Ela recebe os tempos de processamento e o critério de parada, e devolve a **tabela de simulação** (a mesma do quadro).Sobre a **fila de eventos**: ela é uma lista de pares `(tempo, tipo)`. A cada passo, ordenamos por tempo e pegamos o primeiro. Quando dois eventos têm o **mesmo tempo**, precisamos de uma regra de desempate — em sala, adotamos a prioridade `T1` antes de `T2` antes de `T3`. É só uma convenção, mas precisa ser fixada para a tabela ser reproduzível.

In [ ]:
def simular_linha_montagem(tempos_M1, tempos_M2, tempos_M3, parada_relogio=9):
    """
    Simula a linha de montagem de 3 maquinas em serie.

    Parametros
    ----------
    tempos_M1, tempos_M2, tempos_M3 : listas de tempos de processamento
    parada_relogio : a simulacao para quando o relogio ultrapassaria este valor

    Retorna
    -------
    DataFrame com colunas Tempo, Evento, L2, L3 (a tabela de simulacao)
    """
    # --- estado inicial ---
    relogio = 0
    L2, L3 = 0, 0

    # indices que indicam qual sera o proximo tempo a consumir de cada lista
    i1 = i2 = i3 = 0

    # prioridade de desempate quando dois eventos ocorrem no mesmo tempo
    prioridade = {"T1": 1, "T2": 2, "T3": 3}

    # fila de eventos: lista de pares (tempo, tipo)
    # no inicio, so sabemos que M1 vai terminar a 1a unidade
    fila_eventos = [(tempos_M1[i1], "T1")]
    i1 += 1

    # tabela de simulacao: comeca com o estado no tempo 0
    tabela = [(0, "-", L2, L3)]

    # --- ciclo principal ---
    while fila_eventos:
        # 1) proximo evento = menor tempo; desempate pela prioridade do tipo
        fila_eventos.sort(key=lambda ev: (ev[0], prioridade[ev[1]]))
        tempo, tipo = fila_eventos[0]

        # criterio de parada: se o proximo evento passa do limite, encerra
        if tempo > parada_relogio:
            break

        # 2) avanca o relogio e remove o evento da fila
        fila_eventos.pop(0)
        relogio = tempo

        # 3) dispara o evento: atualiza estado e agenda eventos futuros
        if tipo == "T1":
            M2_estava_ociosa = (L2 == 0)
            L2 = L2 + 1
            # M1 sempre tem insumo: agenda o proximo termino de M1
            fila_eventos.append((relogio + tempos_M1[i1], "T1")); i1 += 1
            # se M2 estava livre, esta unidade comeca a ser processada agora
            if M2_estava_ociosa:
                fila_eventos.append((relogio + tempos_M2[i2], "T2")); i2 += 1

        elif tipo == "T2":
            havia_fila_em_M2 = (L2 > 1)
            M3_estava_ociosa = (L3 == 0)
            L2 = L2 - 1
            L3 = L3 + 1
            # se ainda ha unidade esperando em M2, ela entra em processamento
            if havia_fila_em_M2:
                fila_eventos.append((relogio + tempos_M2[i2], "T2")); i2 += 1
            # se M3 estava livre, a unidade que chegou comeca a ser processada
            if M3_estava_ociosa:
                fila_eventos.append((relogio + tempos_M3[i3], "T3")); i3 += 1

        elif tipo == "T3":
            havia_fila_em_M3 = (L3 > 1)
            L3 = L3 - 1
            if havia_fila_em_M3:
                fila_eventos.append((relogio + tempos_M3[i3], "T3")); i3 += 1

        # 4) registra o novo estado na tabela
        tabela.append((relogio, tipo, L2, L3))

    return pd.DataFrame(tabela, columns=["Tempo", "Evento", "L2", "L3"])

Vamos rodar e comparar com a tabela construída no quadro:

In [ ]:
tabela_montagem = simular_linha_montagem(tempos_M1, tempos_M2, tempos_M3, parada_relogio=9)
tabela_montagem

O resultado deve ser **idêntico** à tabela dos slides:| Tempo | Evento | L2 | L3 ||---|---|---|---|| 0 | - | 0 | 0 || 2 | T1 | 1 | 0 || 4 | T1 | 2 | 0 || 5 | T2 | 1 | 1 || 6 | T1 | 2 | 1 || 7 | T3 | 2 | 0 || 9 | T1 | 3 | 0 || 9 | T2 | 2 | 1 |A simulação parou porque qualquer evento seguinte ultrapassaria o relógio = 9.

---## Parte 3 — Estatísticas a partir da tabelaTer a tabela não basta: o objetivo de uma simulação é **extrair indicadores de desempenho**. Vamos calcular dois.Um princípio vale para **toda** estatística baseada em tempo: o estado registrado em uma linha vale **durante o intervalo até a próxima linha**. Por isso, ao percorrer a tabela, ponderamos sempre o **estado anterior** pelo intervalo de tempo decorrido.

### 3.1 — Taxa de ocupação de $M_2$A máquina $M_2$ está **ocupada** sempre que houver pelo menos uma unidade nela ou em sua fila, isto é, $L_2 \\geq 1$.A taxa de ocupação é a fração do tempo total em que $M_2$ esteve ocupada.

In [ ]:
def taxa_ocupacao_M2(tabela):
    """Fração do tempo simulado em que M2 esteve ocupada (L2 >= 1)."""
    tempo_ocupada = 0
    for k in range(1, len(tabela)):
        intervalo   = tabela["Tempo"].iloc[k] - tabela["Tempo"].iloc[k - 1]
        L2_anterior = tabela["L2"].iloc[k - 1]   # estado durante o intervalo
        if L2_anterior >= 1:
            tempo_ocupada += intervalo
    tempo_total = tabela["Tempo"].iloc[-1] - tabela["Tempo"].iloc[0]
    return tempo_ocupada, tempo_total

ocupada, total = taxa_ocupacao_M2(tabela_montagem)
print(f"Tempo com M2 ocupada : {ocupada}")
print(f"Tempo total simulado : {total}")
print(f"Taxa de ocupacao M2  : {ocupada}/{total} = {ocupada/total:.4f}  ({100*ocupada/total:.1f}%)")

Confere com o valor obtido em sala: $7/9 \approx 78\%$.

### 3.2 — Tamanho médio da fila de $M_2$Quando $L_2 \geq 1$, uma unidade está **sendo processada** e as demais ($L_2 - 1$) estão **esperando na fila**. Logo o número de unidades na fila de $M_2$ é $\max(L_2 - 1,\ 0)$.O **tamanho médio da fila** é a média **no tempo** dessa quantidade: a área sob a curva dividida pelo horizonte de tempo. Esta é a estatística do tipo $L_q$.

In [ ]:
def fila_media_M2(tabela):
    """Numero medio de unidades na fila de M2 (media no tempo)."""
    area = 0   # integral de (unidades na fila) ao longo do tempo
    for k in range(1, len(tabela)):
        intervalo   = tabela["Tempo"].iloc[k] - tabela["Tempo"].iloc[k - 1]
        L2_anterior = tabela["L2"].iloc[k - 1]
        na_fila     = max(L2_anterior - 1, 0)
        area += na_fila * intervalo
    tempo_total = tabela["Tempo"].iloc[-1] - tabela["Tempo"].iloc[0]
    return area, tempo_total

area, total = fila_media_M2(tabela_montagem)
print(f"Area acumulada na fila de M2 : {area}")
print(f"Tempo total simulado         : {total}")
print(f"Tamanho medio da fila de M2  : {area}/{total} = {area/total:.4f} unidades")

> **Nota conceitual.** Em sala, calculamos um "tempo médio na fila" dividindo a área acumulada (4) pelo número de unidades que saíram de $M_1$ menos 1. Aquilo é uma estimativa do **tempo por unidade** ($W_q$). Aqui calculamos o **número médio de unidades na fila** ($L_q$), dividindo a mesma área pelo **horizonte de tempo**. São indicadores diferentes — voltaremos a essa distinção na Parte 5, com o exemplo do banco.

---## Parte 4 — Exercício do banco (agência M/M/1)Agora o segundo exemplo da aula:- Uma agência bancária com **1 caixa** e **fila única**.- O tempo **entre chegadas** de clientes segue uma **Exponencial** de média 0,25 h (15 min).- O tempo **de atendimento** também segue uma **Exponencial** de média 0,25 h.**Variável de estado:**- $L$: número de clientes **dentro da agência** (em atendimento + na fila).**Eventos:**| Evento | Significado ||---|---|| `Ch` | chegada de um cliente || `T`  | término de atendimento (cliente sai) |

### 4.1 — Os tempos do exercícioEm sala, usamos uma tabela de números aleatórios já gerada a partir das exponenciais. Vamos reutilizar **os mesmos valores** para reproduzir o gabarito.Mais adiante, no item 4.4, mostramos como **gerar** esses tempos com Python — esse é o elo com a aula de distribuições.

In [ ]:
# Tempos do exercicio (mesmos valores da tabela usada em sala)
tempos_entre_chegadas = [0.044, 0.307, 0.403, 0.119, 0.557, 0.509, 1.324, 0.038, 0.485]
tempos_de_atendimento = [1.331, 0.495, 0.975, 0.011, 0.201, 0.571, 0.242, 0.429, 0.321]

### 4.2 — O motor de simulação do bancoA estrutura é a mesma da linha de montagem. As diferenças:- o critério de parada agora é por **número de eventos** (paramos no evento nº 8), e não pelo relógio;- registramos também o **contador de eventos**, porque ele é o critério de parada;- a regra de cada evento segue a tabela do gabarito:  - `Ch`: $L \leftarrow L + 1$; agenda a próxima chegada; **se o caixa estava livre** ($L = 0$ antes), agenda o término desse cliente;  - `T`: $L \leftarrow L - 1$; **se ainda há cliente esperando** ($L \geq 1$ depois), agenda o término do próximo.

In [ ]:
def simular_banco(tempos_entre_chegadas, tempos_de_atendimento, parada_eventos=8):
    """
    Simula uma agencia M/M/1 (1 caixa, fila unica).

    Parametros
    ----------
    tempos_entre_chegadas : tempos entre chegadas sucessivas de clientes
    tempos_de_atendimento : duracoes de atendimento, na ordem de uso
    parada_eventos : a simulacao para apos processar este numero de eventos

    Retorna
    -------
    DataFrame com colunas Evento_n, Relogio, Evento, L
    """
    # --- estado inicial ---
    relogio = 0.0
    L = 0
    i_ch = i_at = 0          # indices das listas de tempos
    contador = 0             # numero de eventos ja processados

    # fila de eventos: a primeira chegada ja e conhecida no tempo 0
    fila_eventos = [(tempos_entre_chegadas[i_ch], "Ch")]
    i_ch += 1

    tabela = [(0, 0.0, "-", L)]

    # --- ciclo principal ---
    while fila_eventos and contador < parada_eventos:
        # proximo evento = menor tempo
        fila_eventos.sort(key=lambda ev: ev[0])
        relogio, tipo = fila_eventos.pop(0)
        contador += 1

        if tipo == "Ch":
            caixa_estava_livre = (L == 0)
            L = L + 1
            # agenda a proxima chegada
            fila_eventos.append((relogio + tempos_entre_chegadas[i_ch], "Ch"))
            i_ch += 1
            # se o caixa estava livre, este cliente ja comeca a ser atendido
            if caixa_estava_livre:
                fila_eventos.append((relogio + tempos_de_atendimento[i_at], "T"))
                i_at += 1

        elif tipo == "T":
            L = L - 1
            # se ainda ha cliente na fila, o proximo entra em atendimento
            if L >= 1:
                fila_eventos.append((relogio + tempos_de_atendimento[i_at], "T"))
                i_at += 1

        tabela.append((contador, relogio, tipo, L))

    return pd.DataFrame(tabela, columns=["Evento_n", "Relogio", "Evento", "L"])

In [ ]:
tabela_banco = simular_banco(tempos_entre_chegadas, tempos_de_atendimento, parada_eventos=8)
tabela_banco

Compare com o gabarito da aula:| Evento_n | Relógio | Evento | L ||---|---|---|---|| 0 | 0,000 | - | 0 || 1 | 0,044 | Ch | 1 || 2 | 0,351 | Ch | 2 || 3 | 0,754 | Ch | 3 || 4 | 0,873 | Ch | 4 || 5 | 1,375 | T | 3 || 6 | 1,430 | Ch | 4 || 7 | 1,870 | T | 3 || 8 | 1,939 | Ch | 4 |

---## Parte 5 — Estatísticas do banco: $L_q$ e $W_q$ não são a mesma coisaEste é o ponto mais delicado da aula. Existem **duas** médias ligadas à fila, e elas respondem a **perguntas diferentes**:| | $L_q$ — número médio na fila | $W_q$ — tempo médio na fila ||---|---|---|| **O que mede** | quantas pessoas, em média, estavam na fila | quanto tempo, em média, cada cliente esperou || **Tipo de média** | média **no tempo** | média **por cliente** || **Numerador** | área sob a curva de pessoas na fila | soma das esperas individuais || **Denominador** | horizonte de **tempo** | **contagem** de clientes |O erro comum é misturar as duas: usar o numerador de $L_q$ (a área) com o denominador de $W_q$ (a contagem de clientes). O resultado não é nem um nem outro.

### 5.1 — $L_q$: número médio de clientes na filaCom 1 caixa, quando $L \geq 1$ há 1 cliente em atendimento e $L - 1$ na fila. O número na fila é $\max(L - 1,\ 0)$.$L_q$ é a média **no tempo** dessa quantidade.

In [ ]:
def Lq_banco(tabela):
    """Numero medio de clientes na fila (media no tempo)."""
    area = 0.0
    for k in range(1, len(tabela)):
        intervalo  = tabela["Relogio"].iloc[k] - tabela["Relogio"].iloc[k - 1]
        L_anterior = tabela["L"].iloc[k - 1]
        na_fila    = max(L_anterior - 1, 0)
        area += na_fila * intervalo
    horizonte = tabela["Relogio"].iloc[-1] - tabela["Relogio"].iloc[0]
    return area, horizonte

area, horizonte = Lq_banco(tabela_banco)
print(f"Area acumulada na fila : {area:.4f}")
print(f"Horizonte de tempo     : {horizonte:.4f} h")
print(f"Lq = {area:.4f} / {horizonte:.4f} = {area/horizonte:.4f} clientes")

### 5.2 — $W_q$: tempo médio que um cliente espera na filaAqui precisamos acompanhar **cada cliente**: sua espera é o intervalo entre o instante em que **chega** e o instante em que **começa a ser atendido**.Como a fila é única e o atendimento é FIFO (primeiro a chegar, primeiro a ser atendido), o cliente $i$ começa a ser atendido em:$$\text{início}_i = \max(\text{chegada}_i,\ \text{fim do atendimento}_{i-1})$$Um cuidado importante: a simulação parou no evento 8. Os clientes que **ainda estavam na fila** nesse momento têm espera **incompleta** — não sabemos quanto ainda vão esperar. Eles **não entram** no cálculo de $W_q$.

In [ ]:
def Wq_banco(tempos_entre_chegadas, tempos_de_atendimento, horizonte):
    """
    Tempo medio na fila (media por cliente).
    So entram os clientes cuja espera TERMINOU dentro do horizonte simulado.
    """
    instantes_chegada = np.cumsum(tempos_entre_chegadas)

    inicio_atendimento = []
    espera = []
    fim_anterior = 0.0
    for i in range(len(instantes_chegada)):
        inicio = max(instantes_chegada[i], fim_anterior)
        inicio_atendimento.append(inicio)
        espera.append(inicio - instantes_chegada[i])
        fim_anterior = inicio + tempos_de_atendimento[i]

    # mantem apenas quem comecou a ser atendido ate o horizonte
    esperas_completas = [espera[i] for i in range(len(espera))
                         if inicio_atendimento[i] <= horizonte]

    detalhe = pd.DataFrame({
        "Cliente"        : np.arange(1, len(instantes_chegada) + 1),
        "Chegada"        : instantes_chegada,
        "Inicio_atend"   : inicio_atendimento,
        "Espera"         : espera,
        "Espera_completa": [inicio_atendimento[i] <= horizonte
                            for i in range(len(espera))]
    })
    return esperas_completas, detalhe

esperas, detalhe = Wq_banco(tempos_entre_chegadas, tempos_de_atendimento, horizonte)
detalhe

In [ ]:
soma_esperas = sum(esperas)
n_clientes   = len(esperas)
print(f"Esperas consideradas (completas) : {[round(float(e), 3) for e in esperas]}")
print(f"Soma das esperas                 : {soma_esperas:.4f}")
print(f"Numero de clientes               : {n_clientes}")
print(f"Wq = {soma_esperas:.4f} / {n_clientes} = {soma_esperas/n_clientes:.4f} h por cliente")

### 5.3 — Comparando os dois resultadosOs dois números são **diferentes** e isso é esperado: medem coisas distintas. Eles se relacionam pela **Lei de Little**:$$L_q = \lambda \cdot W_q$$onde $\lambda$ é a taxa de chegada. Eles **não** se calculam com a mesma divisão.

In [ ]:
print("Resumo das estatisticas do banco")
print("-" * 48)
print(f"Lq (numero medio na fila)  = {area/horizonte:.4f} clientes")
print(f"Wq (tempo medio na fila)   = {soma_esperas/n_clientes:.4f} h")
print()
print("Observe que sao numeros diferentes: um e media no tempo,")
print("o outro e media por cliente.")

---## Parte 6 — Por que isso ainda não basta: o caminho para o SimPyToda a simulação acima parou em **8 eventos** (banco) ou em **relógio = 9** (linha de montagem). São amostras **curtíssimas**. Os indicadores que calculamos são estimativas, mas **muito instáveis** — se trocássemos os números aleatórios, obteríamos valores bem diferentes.Para ter estimativas confiáveis, precisamos:1. **simular por muito mais tempo** (milhares de eventos), e2. **repetir** a simulação várias vezes, com sementes diferentes, e tomar a média.Fazer isso "na mão", com listas e índices, fica rapidamente inviável quando o sistema tem vários recursos, filas e regras. É exatamente para isso que existem **bibliotecas de simulação**.A partir da próxima aula, usaremos o **SimPy**. Ele cuida automaticamente do relógio, da fila de eventos e do agendamento — e nós passamos a descrever o sistema em termos de **processos** e **recursos**, num nível mais alto.Mas a mecânica por baixo é **exatamente esta** que implementamos aqui: encontrar o próximo evento, avançar o relógio, disparar, agendar, repetir.

### Exercício propostoPegue o **motor da linha de montagem** (`simular_linha_montagem`) e modifique-o para o cenário discutido em sala:> E se os insumos **não** estivessem sempre disponíveis, e o intervalo entre chegadas de insumo seguisse uma distribuição?Dicas:- será preciso uma nova variável de estado, $L_1$ (unidades em $M_1$ + sua fila);- será preciso um novo evento, `T0` (chegada de um insumo no sistema);- a regra de `T1` muda: $M_1$ só inicia uma nova unidade se houver insumo disponível.Compare a tabela resultante com a discussão feita no quadro.

In [ ]:
# Espaco para o exercicio
